In [1]:
import pandas as pd

In [2]:
import openpyxl

In [3]:
landings = pd.read_excel("landings_2026_full_year.xlsx")

In [4]:
Minimum_kg = 0

In [5]:
Maximum_kg = 1000

In [6]:
SPECIES_FIXES = {"tassi":"tasi", "sardines":"dagaa", "chang'u":"changu", "tuna fish":"tuna"}

In [7]:
SITE_FIXES = {"Kizimakazi":"Kizimkazi", "Mkokotoni.":"Mkokotoni", "Shungii":"Shungi"}

In [8]:
YEAR = 2026

In [9]:
MAX_GAP_DAYS = 30

In [10]:
landings.shape

(300, 4)

In [11]:
landings.dtypes

date               str
landing_site       str
species            str
weight_kg       object
dtype: object

In [12]:
landings.head()

,date,landing_site,species,weight_kg
0,02.01.2026,Malindi,kolekole,253.5
1,07.01.2026,Malindi,Changu,59.6
2,07.01.2026,Nungwi,pweza,47.1
3,06.01.2026,Malindi,tuna,199.6
4,06.01.2026,Chwaka,ngisi,45.6


In [13]:
landings.tail()

,date,landing_site,species,weight_kg
295,27.12.2026,Kiongwe,ngisi,72.5
296,27.12.2026,Kikadini,tasi,46.4
297,27.12.2026,Kizimkazi,tasi,61.5
298,30.12.2026,Mkokotoni,changu,37.8
299,31.12.2026,Kikadini,dagaa,374


In [14]:
landings["species"].value_counts()

species
tuna         41
ngisi        41
dagaa        40
kolekole     37
tasi         35
panga        33
pweza        30
changu       29
tassi         2
sardines      2
Changu        1
Panga         1
Tasi          1
Tuna          1
Kolekole      1
dagaa         1
chang'u       1
pweza         1
tuna          1
Tuna fish     1
Name: count, dtype: int64

In [15]:
landings["landing_site"].value_counts()

landing_site
Mkokotoni        40
Nungwi           36
Malindi          33
Fumba            29
Chwaka           23
Kizimkazi        22
Kiongwe          21
Mshembe          14
Kidiko_walevi    13
Vuu              12
Shungi           12
Kilindi          11
Kikadini         10
Kivule            4
Kizimakazi        3
Mkokotoni.        1
 Fumba            1
vuu               1
Kiongwe           1
Kikadini          1
Chwaka            1
malindi           1
kiongwe           1
 Kivule           1
Mkokotoni         1
MKOKOTONI         1
Fumba             1
kizimkazi         1
shungi            1
fumba             1
NUNGWI            1
Shungii           1
Name: count, dtype: int64

In [16]:
weights_raw = pd.to_numeric(landings["weight_kg"], errors="coerce")

In [17]:
landings["weight_kg"].dtype

dtype('O')

In [18]:
landings.loc[[129, 216], "weight_kg"]

129       10,3
216    81.2 kg
Name: weight_kg, dtype: object

In [19]:
weights_raw

0      253.5
1       59.6
2       47.1
3      199.6
4       45.6
       ...  
295     72.5
296     46.4
297     61.5
298     37.8
299    374.0
Name: weight_kg, Length: 300, dtype: float64

In [20]:
failed = weights_raw.isna()

In [21]:
landings[failed]

,date,landing_site,species,weight_kg
76,02.04.2026,Mkokotoni,tuna,abc
83,11.04.2026,Nungwi,ngisi,NaN
95,28.04.2026,Kizimkazi,kolekole,abc
129,06.06.2026,Kikadini,changu,"10,3"
146,30.06.2026,Chwaka,pweza,NaN
155,10.07.2026,Kivule,tasi,NaN
202,25.09.2026,Mkokotoni,panga,NaN
207,30.09.2026,Nungwi,changu,NaN
215,07.10.2026,Vuu,pweza,NaN
216,07.10.2026,Nungwi,ngisi,81.2 kg


In [22]:
landings[weights_raw > Maximum_kg]

,date,landing_site,species,weight_kg
73,30.03.2026,Kiongwe,ngisi,37100
94,30.04.2026,Chwaka,tasi,41000


In [23]:
landings[weights_raw < Minimum_kg]

,date,landing_site,species,weight_kg
84,13.04.2026,Fumba,dagaa,-306
178,20.08.2026,Kivule,pweza,-31.9
181,26.08.2026,Kidiko_walevi,tuna,-204.7


In [24]:
landings["species"] = landings["species"].str.strip().str.lower().replace(SPECIES_FIXES)

In [25]:
landings["species"].value_counts()

species
tuna        44
dagaa       43
ngisi       41
kolekole    38
tasi        38
panga       34
changu      31
pweza       31
Name: count, dtype: int64

In [26]:
landings["landing_site"] = landings["landing_site"].str.strip().str.capitalize().replace(SITE_FIXES)

In [27]:
landings["landing_site"].value_counts()

landing_site
Mkokotoni        43
Nungwi           37
Malindi          34
Fumba            32
Kizimkazi        26
Chwaka           24
Kiongwe          23
Mshembe          14
Shungi           14
Vuu              13
Kidiko_walevi    13
Kikadini         11
Kilindi          11
Kivule            5
Name: count, dtype: int64

In [28]:
dates = pd.to_datetime(landings["date"], format="%d.%m.%Y", errors="coerce")

In [29]:
landings[dates.isna()]

,date,landing_site,species,weight_kg
30,04.02.26,Mkokotoni,changu,36.5
74,2026-03-29,Kidiko_walevi,tasi,61.8
136,12/06/2026,Chwaka,panga,170.2
137,13/06/2026,Kiongwe,dagaa,370
142,27.06.26,Kizimkazi,panga,71.7
143,31.06.2026,Kiongwe,dagaa,248
199,2026-09-20,Fumba,tuna,135.6
255,19/11/2026,Kidiko_walevi,tuna,226.9
262,31.11.2026,Fumba,dagaa,355
264,2026-11-29,Kizimkazi,tasi,42.9


In [30]:
landings[(dates.dt.year != YEAR) & dates.notna()]

,date,landing_site,species,weight_kg
85,12.04.2062,Mkokotoni,kolekole,124.9


In [31]:
gaps = (dates - dates.shift(1)).dt.days

In [32]:
gaps

0      NaN
1      5.0
2      0.0
3     -1.0
4      0.0
      ... 
295    1.0
296    0.0
297    0.0
298    3.0
299    1.0
Name: date, Length: 300, dtype: float64

In [33]:
gaps.abs() > MAX_GAP_DAYS

0      False
1      False
2      False
3      False
4      False
       ...  
295    False
296    False
297    False
298    False
299    False
Name: date, Length: 300, dtype: bool

In [34]:
landings[(gaps > MAX_GAP_DAYS)]

,date,landing_site,species,weight_kg
9,01.10.2026,Nungwi,tasi,58.4
85,12.04.2062,Mkokotoni,kolekole,124.9
153,06.07.2026,Fumba,kolekole,121.4


In [35]:
landings[(gaps.abs() > MAX_GAP_DAYS)]

,date,landing_site,species,weight_kg
9,01.10.2026,Nungwi,tasi,58.4
10,11.01.2026,Mkokotoni,pweza,75.6
85,12.04.2062,Mkokotoni,kolekole,124.9
86,11.04.2026,Mkokotoni,kolekole,160.6
152,07.04.2026,Shungi,ngisi,40.1
153,06.07.2026,Fumba,kolekole,121.4


In [36]:
landings.assign(gap=gaps).iloc[7:12]

,date,landing_site,species,weight_kg,gap
7,09.01.2026,Kikadini,changu,46.4,3.0
8,08.01.2026,Malindi,kolekole,192.7,-1.0
9,01.10.2026,Nungwi,tasi,58.4,266.0
10,11.01.2026,Mkokotoni,pweza,75.6,-263.0
11,12.01.2026,Vuu,dagaa,449,1.0


In [37]:
landings[landings.duplicated()]

,date,landing_site,species,weight_kg
67,27.03.2026,Nungwi,dagaa,496
111,14.05.2026,Mshembe,dagaa,223
134,09.06.2026,Fumba,tasi,14
149,01.07.2026,Mkokotoni,ngisi,20.9
214,07.10.2026,Nungwi,pweza,28.3
270,03.12.2026,Fumba,tuna,224.2


In [38]:
weights_clean = pd.to_numeric((landings["weight_kg"]).astype(str).str.replace("kg","").str.replace(",","."), errors="coerce")

In [39]:
landings["weight_kg"] = weights_clean

In [40]:
landings.loc[[129, 216], "weight_kg"]

129    10.3
216    81.2
Name: weight_kg, dtype: float64

In [41]:
weights_clean.isna().sum()

np.int64(9)

In [42]:
landings[weights_clean.isna()]

,date,landing_site,species,weight_kg
76,02.04.2026,Mkokotoni,tuna,NaN
83,11.04.2026,Nungwi,ngisi,NaN
95,28.04.2026,Kizimkazi,kolekole,NaN
146,30.06.2026,Chwaka,pweza,NaN
155,10.07.2026,Kivule,tasi,NaN
202,25.09.2026,Mkokotoni,panga,NaN
207,30.09.2026,Nungwi,changu,NaN
215,07.10.2026,Vuu,pweza,NaN
219,12.10.2026,Fumba,tuna,NaN


In [43]:
weights_clean

0      253.5
1       59.6
2       47.1
3      199.6
4       45.6
       ...  
295     72.5
296     46.4
297     61.5
298     37.8
299    374.0
Name: weight_kg, Length: 300, dtype: float64

In [44]:
landings["note"] = ""

In [45]:
grams = weights_clean > Maximum_kg

In [46]:
landings[grams]

,date,landing_site,species,weight_kg,note
73,30.03.2026,Kiongwe,ngisi,37100.0,
94,30.04.2026,Chwaka,tasi,41000.0,


In [47]:
landings.loc[grams, "weight_kg"] = landings.loc[grams, "weight_kg"]/1000

In [48]:
landings.loc[grams, "weight_kg"]

73    37.1
94    41.0
Name: weight_kg, dtype: float64

In [49]:
landings.loc[grams, "note"] += "g to Kg; "

In [50]:
landings[grams]

,date,landing_site,species,weight_kg,note
73,30.03.2026,Kiongwe,ngisi,37.1,g to Kg;
94,30.04.2026,Chwaka,tasi,41.0,g to Kg;


In [51]:
negatives = weights_clean < Minimum_kg

In [52]:
landings[negatives]

,date,landing_site,species,weight_kg,note
84,13.04.2026,Fumba,dagaa,-306.0,
178,20.08.2026,Kivule,pweza,-31.9,
181,26.08.2026,Kidiko_walevi,tuna,-204.7,


In [53]:
landings.loc[negatives, "weight_kg"] = landings.loc[negatives, "weight_kg"] * -1

In [54]:
landings.loc[negatives, "weight_kg"]

84     306.0
178     31.9
181    204.7
Name: weight_kg, dtype: float64

In [55]:
landings.loc[negatives, "note"] += "-ve to +ve; "

In [56]:
landings[negatives]

,date,landing_site,species,weight_kg,note
84,13.04.2026,Fumba,dagaa,306.0,-ve to +ve;
178,20.08.2026,Kivule,pweza,31.9,-ve to +ve;
181,26.08.2026,Kidiko_walevi,tuna,204.7,-ve to +ve;


In [57]:
landings[landings["note"] != ""]

,date,landing_site,species,weight_kg,note
73,30.03.2026,Kiongwe,ngisi,37.1,g to Kg;
84,13.04.2026,Fumba,dagaa,306.0,-ve to +ve;
94,30.04.2026,Chwaka,tasi,41.0,g to Kg;
178,20.08.2026,Kivule,pweza,31.9,-ve to +ve;
181,26.08.2026,Kidiko_walevi,tuna,204.7,-ve to +ve;


In [58]:
landings.loc[[129, 216], "weight_kg"]

129    10.3
216    81.2
Name: weight_kg, dtype: float64

In [59]:
landings["weight_kg"].dtype

dtype('float64')

In [60]:
landings["weight_kg"].isna().sum()

np.int64(9)